# TSMC SEC 月营收原件身份与 NAS 留存

只读复用既有 Qlib 来源卡；为后续 Stage1 信息检验核验 131 份原件和历史支持，归档原字节，不读取 SOXX 收益。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex
import pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_tsmc_monthly_source_retention_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();source=root/plan['source_root'];old=root/'research/tsmc_monthly_revenue_pit_qualification_20261010'
assert plan['frozen'] and sha(source/'docs_manifest.json')==plan['docs_manifest_sha256'] and sha(source/'submissions.json')==plan['submissions_sha256'] and sha(root/plan['source_summary'])==plan['source_summary_sha256']
recpath=old/'qlib_recorder'/plan['source_experiment_id']/plan['source_recorder_id']/'artifacts/events';assert sha(recpath)==plan['source_events_artifact_sha256']
manifest=json.loads((source/'docs_manifest.json').read_text());assert len(manifest)==131;bad=[]
for row in manifest:
 p=source/'docs'/(row['accn']+'_'+pathlib.Path(row['url']).name)
 if not p.is_file() or sha(p)!=row['sha256']:bad.append(row['accn'])
assert not bad
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
with R.uri_context(str(old/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=plan['source_experiment_id'],recorder_id=plan['source_recorder_id']);assert rec.status=='FINISHED';events=rec.load_object('events');docs=rec.load_object('docs_manifest')
assert len(events)==131 and len(docs)==131 and int(events.admitted_to_scope.sum())==130
cal=pd.DatetimeIndex(D.calendar(start_time='2015-12-11',end_time='2026-09-25',freq='day'));frame=D.features(['TSMC_RELEASE'],['$acceleration_pct_points','$event_available'],start_time='2015-12-11',end_time='2026-09-25',freq='day').reset_index()
assert len(cal)==len(frame)==2712 and int(frame['$event_available'].sum())==130
support={'2016_2022_original_events':int(events.loc[events.factor_eligible_session.notna()&events.period.between('2016-01','2022-12')].shape[0]),'2023_2025_original_events':int(events.loc[events.factor_eligible_session.notna()&events.period.between('2023-01','2025-12')].shape[0]),'calendar_rows':len(cal),'event_available_rows':int(frame['$event_available'].sum()),'first_feature_date':str(frame.datetime.min()),'last_feature_date':str(frame.datetime.max())}
files=sorted(p for p in source.rglob('*') if p.is_file());assert len(files)==264;members={str(p.relative_to(root)):{'sha256':sha(p),'bytes':p.stat().st_size} for p in files}
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-source-retention-') as temporary:
 archive=pathlib.Path(temporary)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0];assert remote==digest
 archive_receipt={**receipt,'nas_path':remote_path,'source_files':len(files),'source_bytes':sum(x['bytes'] for x in members.values())}
source_result={'status':'existing_SEC_original_monthly_source_NAS_backed','source_archive':archive_receipt,'source_members_manifest_sha256':hashlib.sha256(json.dumps(members,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'source_member_count':len(members),'support':support,'old_recorder_id':plan['source_recorder_id'],'old_experiment_id':plan['source_experiment_id'],'new_model_fits':0,'new_accounts':0,'SOXX_outcome_reads':0};(card/'result.json').write_text(json.dumps(source_result,ensure_ascii=False,indent=2)+'\n')
rec_evidence=archive_nas_recorder(card/'nas_recorder_binding.json',workspace_root=root,card=old,recorder_id=plan['source_recorder_id'],experiment_id=plan['source_experiment_id']);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(rec_evidence['nas_path'])).split()[0]==rec_evidence['sha256']
with tempfile.TemporaryDirectory(prefix='tsmc-source-cold-') as temporary:
 source_copy=pathlib.Path(temporary)/'source.tar.gz';rec_copy=pathlib.Path(temporary)/'recorder.tar.gz'
 for remote,target,expected,bytes_expected in [(remote_path,source_copy,digest,archive_receipt['size_bytes']),(rec_evidence['nas_path'],rec_copy,rec_evidence['sha256'],rec_evidence['size_bytes'])]:
  with target.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True)
  assert target.stat().st_size==bytes_expected and sha(target)==expected
 with tarfile.open(source_copy,'r:gz') as t:
  names={m.name for m in t if m.isfile()};assert names==set(members)
  for name in names:
   data=t.extractfile(name).read();assert len(data)==members[name]['bytes'] and hashlib.sha256(data).hexdigest()==members[name]['sha256'] and data==(root/name).read_bytes()
 checked={}
 with tarfile.open(rec_copy,'r:gz') as t:
  for name in ('events','docs_manifest','reference_asof_state'):
   rel=f"qlib_recorder/{plan['source_experiment_id']}/{plan['source_recorder_id']}/artifacts/{name}";data=t.extractfile(rel).read();assert data==(old/rel).read_bytes();checked[name]={'sha256':hashlib.sha256(data).hexdigest(),'bytes':len(data)}
readback={'status':'NAS_all_264_original_source_files_and_selected_existing_Recorder_objects_cold_restored_exact','source_archive_sha256':digest,'source_members_checked':len(members),'Recorder_archive_sha256':rec_evidence['sha256'],'Recorder_members_checked':checked};(card/'nas_scoped_restore.json').write_text(json.dumps(readback,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'TSMC_2015_2026_original_SEC_monthly_source_backed_2016_2022_information_receiver_unrun','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_files':len(files),'source_archive':archive_receipt,'existing_Recorder_archive':rec_evidence,'NAS_scoped_restore':'nas_scoped_restore.json','support':support,'model_fits':0,'native_accounts':0,'SOXX_outcome_reads':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['reuse old Recorder events/docs_manifest/reference_asof_state']}};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'source_sha256':digest,'Recorder_sha256':rec_evidence['sha256'],'source_files':len(files),'support':support},ensure_ascii=False))


## 中文结论

复核既有来源卡：**131** 份 SEC 原报文件逐份哈希与原 `docs_manifest.json` 一致，130个2015-11至2026-08月度事件在原生 Qlib 来源中可读。`D.calendar` 与 `D.features` 在来源截止面有 **2712** 个日位置、**130** 个事件可用日；其中2016–2022有 **84** 个原报月度事件，2023–2025有 **36** 个。沿用 SEC 接收时间之后下一美股交易日的保守时钟，不能改称发行人新闻首发时刻。

本机来源根下全部 **264** 个文件原字节已归档 Yanxuan NAS，归档 SHA-256 `ef85974f40111ea6004efb36be63986734da2203e94408b4ff973df397b1f727`；旧 Qlib FINISHED Recorder `4d1dc18e75c7489d8251af1e737c1a0f` 同步归档，SHA-256 `b82835874dd9a3179980f43847dc4973b4ba32dc19275c954bb92f9a4e67f786`。两份压缩包均从远端限定下载，全部264个来源文件和Recorder中事件、原件清单、Qlib日状态三件对象与本机字节逐项一致。[冷恢复收据](nas_scoped_restore.json)保留校验数和哈希。

决定 `TSMC_2015_2026_original_SEC_monthly_source_backed_2016_2022_information_receiver_unrun`。这是来源可恢复性与训练/留出支持度门，**0 行情结果读取、0 拟合、0 资金账户**；不能据此宣称月营收加速度有SOXX预测力。2011–2015原 Stage1 训练期没有该因子，2026年9月25日之后旧来源面不延伸；未来信息门须保持这些日期未知、冻结接收任务和所有对照，再决定是否值得启动完整0bp账户。
